<a href="https://colab.research.google.com/github/StaryDron/PigPostureComputerVision/blob/main/efficientnet_augmentacja_FDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Instalacja i Kaggle API ───────────────────────────────────────────────────
!pip install albumentations -q
from google.colab import files
import os, shutil

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Wgraj plik kaggle.json:")
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle competitions download -c multi-view-pig-posture-recognition
!unzip -q multi-view-pig-posture-recognition.zip

# ── Importy ───────────────────────────────────────────────────────────────────
import os, ast, copy, re, time, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ── GPU i Stałe ───────────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

CLASS_NAMES = {0: "Lateral_lying_left", 1: "Lateral_lying_right", 2: "Sitting", 3: "Standing", 4: "Sternal_lying"}
NUM_CLASSES = len(CLASS_NAMES)
BASE_DIR    = Path("multiview_pig_posture_recognition")
TRAIN2_IMGS = BASE_DIR / "train2_images"
TEST_IMGS   = BASE_DIR / "test_images"

BATCH_SIZE = 32
EPOCHS     = 6
SAVE_PATH  = "T2_effnetv2_fda_camera_aug.pt"

# ── Przygotowanie danych ──────────────────────────────────────────────────────
def parse_camera_meta(image_id):
    m = re.match(r"(pen\d+)_(orb|tur)_(cam\d+)_", image_id)
    if m: return m.group(1), m.group(2), m.group(3)
    return "unknown", "unknown", "unknown"

def add_camera_cols(df):
    df["pen"]      = df["image_id"].apply(lambda x: parse_camera_meta(x)[0])
    df["cam_type"] = df["image_id"].apply(lambda x: parse_camera_meta(x)[1])
    df["cam_num"]  = df["image_id"].apply(lambda x: parse_camera_meta(x)[2])
    df["camera"]   = df["pen"] + "_" + df["cam_type"] + "_" + df["cam_num"]
    return df

train2 = pd.read_csv(BASE_DIR / "train2.csv")
train2["source"] = "train2"
train2["bbox_parsed"] = train2["bbox"].apply(ast.literal_eval)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / "test.csv")
test["source"] = "test"
test["bbox_parsed"] = test["bbox"].apply(ast.literal_eval)
test = add_camera_cols(test)

def load_image(image_id, source):
    folder = {"train2": TRAIN2_IMGS, "test": TEST_IMGS}[source]
    return Image.open(folder / image_id).convert("RGB")

def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1, y1 = x - w * padding, y - h * padding
    x2, y2 = x + w + w * padding, y + h + h * padding
    if make_square:
        side = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2, y1, y2 = cx - side/2, cx + side/2, cy - side/2, cy + side/2
    x1, y1 = max(0, int(round(x1))), max(0, int(round(y1)))
    x2, y2 = min(img_w, int(round(x2))), min(img_h, int(round(y2)))
    return image.crop((max(x1, 0), max(y1, 0), max(x2, x1+1), max(y2, y1+1)))

# ── FDA Core & Bufor Obrazów ──────────────────────────────────────────────────
def fda_transfer(src_img: np.ndarray, tgt_img: np.ndarray, beta: float = 0.001) -> np.ndarray:
    src = src_img.astype(np.float32) / 255.0
    tgt = tgt_img.astype(np.float32) / 255.0

    if src.shape != tgt.shape:
        tgt_pil = Image.fromarray((tgt * 255).astype(np.uint8)).resize((src.shape[1], src.shape[0]), Image.BILINEAR)
        tgt = np.array(tgt_pil).astype(np.float32) / 255.0

    result = np.zeros_like(src)
    for c in range(src.shape[2]):
        src_fft = np.fft.fft2(src[:, :, c])
        tgt_fft = np.fft.fft2(tgt[:, :, c])
        src_fft_shift = np.fft.fftshift(src_fft)
        tgt_fft_shift = np.fft.fftshift(tgt_fft)

        src_amp = np.abs(src_fft_shift)
        src_pha = np.angle(src_fft_shift)
        tgt_amp = np.abs(tgt_fft_shift)

        h, w = src.shape[:2]
        b_h, b_w = int(np.floor(beta * h)), int(np.floor(beta * w))
        c_h, c_w = h // 2, w // 2

        src_amp_new = src_amp.copy()
        src_amp_new[c_h - b_h: c_h + b_h, c_w - b_w: c_w + b_w] = tgt_amp[c_h - b_h: c_h + b_h, c_w - b_w: c_w + b_w]

        src_fft_new = src_amp_new * np.exp(1j * src_pha)
        src_fft_new = np.fft.ifftshift(src_fft_new)
        result[:, :, c] = np.fft.ifft2(src_fft_new).real

    return (np.clip(result, 0, 1) * 255).astype(np.uint8)

# Ładujemy pulę referencyjną z TESTA (pen2_orb_cam2) do RAM
print("Ładowanie obrazów referencyjnych (test: pen2_orb_cam2) do RAM...")
test_ref_df = test[test["camera"] == "pen2_orb_cam2"]
# Pobieramy próbkę np. 100 obrazów (żeby zróżnicować widmo, a nie zapchać pamięci)
test_ref_df = test_ref_df.sample(min(100, len(test_ref_df)), random_state=42)
TEST_REF_IMAGES = [np.array(load_image(row["image_id"], "test")) for _, row in test_ref_df.iterrows()]

# ── Funkcje Augmentacji ───────────────────────────────────────────────────────
FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

# Klasyczne transformacje dla kamer tur (tak jak w EfficientNet_Augmentacja.ipynb)
def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    return TF.adjust_brightness(img, brightness_factor=0.95)

def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    return TF.adjust_saturation(img, saturation_factor=0.9)

def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    return TF.adjust_saturation(img, saturation_factor=0.85)

# NOWA transformacja FDA tylko dla kamer pen2_orb (zgodnie z życzeniem)
def make_fda_orb_aug(flip, beta=0.001, saturation=0.2):
    def aug_fn(img):
        if flip:
            img = TF.hflip(img)
        src_np = np.array(img)
        tgt_np = random.choice(TEST_REF_IMAGES)
        img_np = fda_transfer(src_np, tgt_np, beta=beta)
        img_pil = Image.fromarray(img_np)
        return TF.adjust_saturation(img_pil, saturation_factor=saturation)
    return aug_fn

CAMERA_AUG_FN = {
    "pen2_tur_cam1": (aug_pen2_tur_cam1, True),
    "pen1_tur_cam2": (aug_pen1_tur_cam2, True),
    "pen2_tur_cam2": (aug_pen2_tur_cam2, False),
    # Dla pen2_orb używamy teraz FDA!
    "pen2_orb_cam1": (make_fda_orb_aug(flip=True, beta=0.001, saturation=0.2), True),
    "pen2_orb_cam2": (make_fda_orb_aug(flip=False, beta=0.001, saturation=0.2), False),
}
# UWAGA: kamery pen1_orb_cam1 i pen1_orb_cam2 nie mają wpisu w słowniku,
# więc przejdą czyste i nietknięte (tylko oryginał).

# ── Dataset i Dataloader ──────────────────────────────────────────────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15): self.std, self.p = std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PigDatasetCameraAug(Dataset):
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.df = df.reset_index(drop=True)
        self.base_transform = base_transform
        self.camera_aug_fn = camera_aug_fn
        self.samples = []
        for idx, row in self.df.iterrows():
            cam = row.get("camera", "unknown")
            label = int(row["class_id"])
            self.samples.append((idx, False, label)) # Zawsze dodajemy oryginał

            # Jeśli w słowniku istnieje augmentacja dla tej kamery
            if is_train and cam in camera_aug_fn:
                _, flip_label = camera_aug_fn[cam]
                new_label = FLIP_LABEL_MAP[label] if flip_label else label
                self.samples.append((idx, True, new_label)) # Zaugmentowany

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        row_idx, do_aug, label = self.samples[idx]
        row = self.df.iloc[row_idx]
        image = load_image(row["image_id"], row["source"])
        crop = crop_with_padding(image, row["bbox_parsed"], padding=0.12, make_square=True)

        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row["camera"]]
            crop = aug_fn(crop)

        return self.base_transform(crop), label

train_dataset = PigDatasetCameraAug(train2, CAMERA_AUG_FN, BASE_TRANSFORM, is_train=True)

all_labels = [s[2] for s in train_dataset.samples]
class_counts = np.bincount(all_labels, minlength=NUM_CLASSES)
sample_weights = np.array([1.0 / max(class_counts[l], 1) for l in all_labels])
sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)

# ── Model, Trening i Zapis ────────────────────────────────────────────────────
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

best_f1 = 0.0
for epoch in range(EPOCHS):
    model.train()
    all_preds, all_labels_tr = [], []
    for batch_i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        all_labels_tr.extend(labels.cpu().numpy())

    tr_f1 = f1_score(all_labels_tr, all_preds, average="macro")
    print(f"Epoch {epoch+1}/{EPOCHS} | Train F1: {tr_f1:.4f}")
    scheduler.step(tr_f1)

    if tr_f1 > best_f1:
        best_f1 = tr_f1
        torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES}, SAVE_PATH)

print("Trening zakończony. Przygotowuję submission...")

# ── Inferencja ────────────────────────────────────────────────────────────────
class PigTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_id"], row["source"])
        crop = crop_with_padding(image, row["bbox_parsed"], padding=0.12, make_square=True)
        return self.transform(crop), row["row_id"]

test_loader = DataLoader(PigTestDataset(test, VAL_TRANSFORM), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
model.load_state_dict(torch.load(SAVE_PATH)["model_state_dict"])
model.eval()

all_row_ids, all_preds_test = [], []
with torch.no_grad():
    for images, row_ids in test_loader:
        preds = model(images.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_row_ids.extend(row_ids)
        all_preds_test.extend(preds)

submission = pd.DataFrame({"row_id": all_row_ids, "class_id": all_preds_test})
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")
submission = submission.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()

submission.to_csv("submission_T2_camera_aug.csv", index=False)
files.download("submission_T2_camera_aug.csv")
files.download(SAVE_PATH)
print("Gotowe!")